In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# โหลด CSV
df = pd.read_csv("jobs_search_master.csv", dtype=str, keep_default_na=False)

# เลือก column สำคัญ
text_col = ["detail","position"]  # ข้อความสำหรับ embedding
metadata_cols = [
    "id","jobpost_id","company_id","occupation_id","company_name", "company_name_eng", "position", "job_province", "salary_start", "salary_end"
]

# สร้าง embeddings
model = SentenceTransformer("paraphrase-multilingual-MPNet-base-v2")
texts = df[text_col].fillna("").tolist()
embeddings = model.encode(texts, show_progress_bar=True).tolist()

# เตรียม metadata
metadatas = df[metadata_cols].to_dict(orient="records")
ids = df[id_col].tolist()

# สร้าง Chroma collection
client = chromadb.HttpClient(
    host=os.getenv("CHROMA_HOST", "localhost"),
    port=int(os.getenv("CHROMA_PORT", "8000")),
    settings=Settings(anonymized_telemetry=False)
)
collection = client.get_or_create_collection(name="jobs_search_master_vector")

# เพิ่มข้อมูลเข้า collection
collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
    embeddings=embeddings
)

print("✅ เพิ่มข้อมูลเข้า Chroma Collection เรียบร้อยแล้ว")


In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# โหลด CSV
df = pd.read_csv("jobs_search_master.csv", dtype=str, keep_default_na=False)

# รวมข้อความจากหลายคอลัมน์สำหรับ embedding
df["combined_text"] = df["detail"].fillna("") + " " + df["position"].fillna("")
texts = df["combined_text"].tolist()

# เตรียม metadata
metadata_cols = [
    "id","jobpost_id","company_id","occupation_id","company_name",
    "company_name_eng","position","job_province","salary_start","salary_end"
]
metadatas = df[metadata_cols].to_dict(orient="records")
ids = df["id"].tolist()

# โหลดโมเดล embedding
model = SentenceTransformer("intfloat/multilingual-e5-large")

# สร้าง embeddings
embeddings = model.encode(texts, show_progress_bar=True).tolist()

# เชื่อมต่อ Chroma
client = chromadb.HttpClient(
    host=os.getenv("CHROMA_HOST", "localhost"),
    port=int(os.getenv("CHROMA_PORT", "8000")),
    settings=Settings(anonymized_telemetry=False)
)

# สร้างหรือโหลด collection
collection = client.get_or_create_collection(name="jobs_search_master_vector02")

# เพิ่มข้อมูลเข้า collection
collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
    embeddings=embeddings
)

print("✅ เพิ่มข้อมูลเข้า Chroma Collection เรียบร้อยแล้ว")


In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# โหลด CSV
df = pd.read_csv("member_resume_search_master.csv", dtype=str, keep_default_na=False)

# รวมข้อความจากหลายคอลัมน์สำหรับ embedding
df["combined_text"] = df["member_resume_hardskill"].fillna("") + " " + df["member_resume_hardskill_other"].fillna("")+ " " + df["member_resume_softskill"].fillna("")
texts = df["combined_text"].tolist()

# เตรียม metadata
metadata_cols = [
    "id","member_resume_id","member_user_id","occupation_new_name","occupation_sub_name",
]
metadatas = df[metadata_cols].to_dict(orient="records")
ids = df["id"].tolist()

# โหลดโมเดล embedding
model = SentenceTransformer("BAAI/bge-multilingual-gemma2")

# สร้าง embeddings
embeddings = model.encode(texts, show_progress_bar=True).tolist()

# เชื่อมต่อ Chroma
client = chromadb.HttpClient(
    host=os.getenv("CHROMA_HOST", "localhost"),
    port=int(os.getenv("CHROMA_PORT", "8000")),
    settings=Settings(anonymized_telemetry=False)
)

# สร้างหรือโหลด collection
collection = client.get_or_create_collection(name="member_resume_search_master_vector01")

# เพิ่มข้อมูลเข้า collection
collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
    embeddings=embeddings
)

print("✅ เพิ่มข้อมูลเข้า Chroma Collection เรียบร้อยแล้ว")


In [ ]:
# ดู collections
collections = client.list_collections()
print(f"Collections: {[c.name for c in collections]}")

In [ ]:
collection = client.get_collection("jobs_search001")  # replace with your collection name
print("Number of items:", collection.count())

In [ ]:
results = collection.get(include=["metadatas"])
print(f"จำนวน metadata: {len(results['metadatas'])}")
print("ตัวอย่าง metadata 5 แถวแรก:")
for i, m in enumerate(results['metadatas'][:5]):
    print(i, m)

In [ ]:
# ลบ collection
client.delete_collection("job_search")
print("✅ ลบ collection สำเร็จ")